In [17]:
import pandas as pd
df = pd.read_csv('pedidos_ecommerce.csv')


In [18]:
# Volume: quantas linhas e colunas?
print(df.shape)

(5000, 14)


In [19]:
# Tipos: o que é número, o que é texto, o que é data?
print(df.dtypes)

pedido_id           object
data_pedido         object
cliente_id          object
cidade              object
estado              object
categoria           object
produto             object
quantidade           int64
preco_unitario     float64
valor_total        float64
desconto           float64
valor_final        float64
forma_pagamento     object
status              object
dtype: object


In [20]:
# Primeiras linhas: como o dado se parece?
df.head(10)

,pedido_id,data_pedido,cliente_id,cidade,estado,categoria,produto,quantidade,preco_unitario,valor_total,desconto,valor_final,forma_pagamento,status
0,PED00001,2024-02-22,CLI9935,Goiânia,GO,Roupas,Camiseta Polo,3,82.04,246.12,0.00,246.12,cartão crédito,cancelado
1,PED00002,2024-11-04,CLI4257,São Paulo,SP,Livros,O Senhor dos Aneis,1,65.62,65.62,0.00,65.62,boleto,em transporte
2,PED00003,2024-01-04,CLI7924,Belo Horizonte,MG,Alimentos,Whey Protein 2kg,2,191.87,383.74,0.00,383.74,cartão débito,entregue
3,PED00004,2024-07-02,CLI5333,Salvador,BA,Calçados,Tenis Nike,3,285.77,857.31,0.00,857.31,cartão crédito,em transporte
4,PED00005,2024-10-09,CLI6925,Porto Alegre,RS,Brinquedos,Carrinho Hot Wheels,1,59.06,59.06,0.00,59.06,pix,entregue
5,PED00006,2024-02-21,CLI5554,Fortaleza,CE,Roupas,Camiseta Polo,2,129.58,259.16,33.25,225.91,pix,em transporte
6,PED00007,2024-12-15,CLI2169,Goiânia,GO,Beleza,Perfume Importado,3,201.71,605.13,63.70,541.43,pix,em transporte
7,PED00008,2024-11-23,CLI4598,Florianópolis,SC,Calçados,Bota Couro,2,249.02,498.04,0.00,498.04,boleto,entregue
8,PED00009,2024-02-03,CLI6155,Curitiba,PR,Eletrônicos,Smartphone Samsung,1,3051.61,3051.61,0.00,3051.61,cartão crédito,em transporte
9,PED00010,2024-10-14,CLI5304,Manaus,AM,Brinquedos,Boneca Barbie,4,97.14,388.56,0.00,388.56,boleto,cancelado


In [21]:
import os



In [22]:
df.to_csv('pedidos.csv', index=False)
df.to_json('pedidos.json', orient='records', lines=True)
df.to_parquet('pedidos.parquet', index=False)

In [29]:
import os

# Veja o tamanho de cada arquivo em disco
# Preste atenção na diferença entre os três
for arquivo in ['pedidos.csv', 'pedidos.json', 'pedidos.parquet']:
    kb = os.path.getsize(arquivo) / 1024
    print(f'{arquivo:30s} {kb:.1f} KB')

pedidos.csv                    564.2 KB
pedidos.json                   1548.8 KB
pedidos.parquet                196.1 KB


Curiosidade: abra o pedidos.json num editor de texto qualquer e compare com o
pedidos.csv. O que você percebe de diferente na forma como o mesmo dado está
escrito?

**Resposta: o JSON possui chave valor, já o CSV não tem estrutura, ele vem separado por , e não tem metadado igual o JSON**

In [40]:
import duckdb
import time
queries = {
'CSV': """SELECT cidade, SUM(valor_final) as total FROM 'pedidos.csv'
GROUP BY cidade ORDER BY total DESC""",
'JSON': """SELECT cidade, SUM(valor_final) as total FROM 'pedidos.json'
GROUP BY cidade ORDER BY total DESC""",
'Parquet': """SELECT cidade, SUM(valor_final) as total FROM
'pedidos.parquet' GROUP BY cidade ORDER BY total DESC""",
}
for nome, query in queries.items():
    inicio = time.time()
    resultado = duckdb.sql(query).df()
    fim = time.time()
    print(f'{nome:10s} {(fim - inicio)*1000:.2f} ms')
# Os três resultados abaixo devem ser idênticos
# O formato não muda o dado — muda como ele é armazenado e lido

CSV        122.43 ms
JSON       95.64 ms
Parquet    3.37 ms


In [41]:
print(resultado)

            cidade      total
0         Salvador  631505.49
1           Recife  628122.98
2     Porto Alegre  584952.14
3   Rio de Janeiro  564292.38
4         Curitiba  558364.43
5           Manaus  528258.95
6            Belém  524250.22
7    Florianópolis  522329.27
8   Belo Horizonte  514351.77
9          Goiânia  487020.52
10       Fortaleza  472879.10
11       São Paulo  470796.93


In [42]:
from IPython.display import display

def dinheiro_br(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

display(
    resultado.style
    .format({"total": dinheiro_br})
    .hide(axis="index")
)

cidade,total
Salvador,"R$ 631.505,49"
Recife,"R$ 628.122,98"
Porto Alegre,"R$ 584.952,14"
Rio de Janeiro,"R$ 564.292,38"
Curitiba,"R$ 558.364,43"
Manaus,"R$ 528.258,95"
Belém,"R$ 524.250,22"
Florianópolis,"R$ 522.329,27"
Belo Horizonte,"R$ 514.351,77"
Goiânia,"R$ 487.020,52"
